In [ ]:
import os, subprocess
subprocess.run("pip install -q scikit-learn pandas statsmodels "numpy<2"", shell=True)
os.makedirs("/workspace/data", exist_ok=True)
os.makedirs("/workspace/outputs/t22", exist_ok=True)


# T22 — PR-alone and RT-alone on CD-HIT-97%, closing Table 11's mixed-run problem

**Why this notebook exists.** Table 11 reports four rows: PR alone, RT alone, PR+RT combined, and
PR+RT under dropout. The PR+RT-combined and dropout rows were regenerated on the CD-HIT-97% split
(T20B/T21) after a mapping bug was fixed that had silently fallen back to the older MinHash grouping.
The PR-alone and RT-alone rows were **never regenerated** -- they are still whatever an earlier run
(T17, on the MinHash split) produced, which is why their standard deviations (0.004-0.006) look
nothing like the two regenerated rows (0.022-0.029) for the same models on what the caption claims
is the same split. A reviewer caught this directly: mixing two different runs under one caption, with
one row (RT alone, 0.9869) scoring higher than another (PR+RT combined, 0.9699) in a way that doesn't
make sense for a k-mer model (more sequence should not hurt it) unless it's an artifact of comparing
two different runs -- which it is. **This notebook regenerates PR alone and RT alone on the CD-HIT-97%
split, three seeds, all five models**, using the exact same pipeline (verified cluster mapper, same
split code, same CNN architectures/training loop) as T20B and T21, so all four rows in Table 11 will
finally come from one consistent run.

**Needs:** your `t11_hxb2_coords.csv` from the earlier T11 run (same file T17, T20B, and T21 use),
attached as a Kaggle Dataset.

**GPU needed:** yes, for the three CNNs. Two inputs (PR alone, RT alone) x 3 seeds x 3 CNNs = 18 CNN
fits, on inputs *shorter* than T21's PR+RT amplicon (PR is ~297 nt, RT is ~720 nt, versus ~1,015 nt for
PR+RT) so each fit is if anything slightly cheaper than T21's. **Roughly 30-45 minutes of GPU time.**

**After this run:** send back the results and the PR/RT rows in Table 11 get replaced with these
numbers, all four rows will be on one run, and the RT-vs-PR+RT question (does adding protease to RT
genuinely lower the score, or was that a same-run-vs-different-run artifact) gets answered directly.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install CD-HIT

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, subprocess, shutil
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/workspace/outputs/t22"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)

import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, GaussianNoise,
    SpatialDropout1D, Layer, Concatenate, MaxPool1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical



### 2. Load the full dataset and run CD-HIT-97% (same command/args as T18/T19/T20/T21, so clusters match)

In [ ]:

DATA_PATH = "/workspace/data/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
X_all = df["sequence"].values
n = len(df)

with open(f"{OUT}/pol_t22.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all()
    return labels

out = f"{OUT}/pol_t22_cdhit_97"
cmd = f"cd-hit-est -i {OUT}/pol_t22.fasta -o {out} -c 0.97 -n 10 -M 16000 -T 4 -d 0"
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
assert r.returncode == 0 and os.path.exists(out + ".clstr"), f"CD-HIT failed: {r.returncode}"
groups_all = parse_cdhit_clusters(out + ".clstr", n)
print(f"CD-HIT 97% identity: {len(set(groups_all))} clusters (should match T18/T19/T20/T21's 7,182)")


### 3. Load the T11 PR/RT coordinates. Inputs this time: PR alone and RT alone (not combined).

In [ ]:

COORDS_PATH = "/workspace/data/t11_hxb2_coords.csv"
assert os.path.exists(COORDS_PATH), (
    f"{COORDS_PATH} not found. This notebook needs your T11 output (t11_hxb2_coords.csv) attached "
    "as a Kaggle Dataset.")

coords_df = pd.read_csv(COORDS_PATH)
good = coords_df.dropna(subset=["pr", "rt"]).reset_index(drop=True)
print(f"{len(good)} sequences with both PR and RT extracted")

le = LabelEncoder(); y_good = le.fit_transform(good["subtype"].values)
n_cls = len(le.classes_)
INPUTS = ["pr", "rt"]   # <-- the two inputs Table 11 still needs on this split

# Identical verified mapper to T20B/T21 -- tries several strategies and refuses to guess if none verify.
def build_amplicon_groups(coords_df, good, df, X_all, groups_all):
    n_full = len(df)
    for col in ["orig_index", "orig_idx", "index", "row_index", "source_index", "full_index"]:
        if col in good.columns:
            try:
                idx = good[col].astype(int).values
                if idx.min() >= 0 and idx.max() < n_full:
                    return groups_all[idx], f"used explicit '{col}' column"
            except Exception:
                pass
    for col in ["sequence", "full_sequence", "seq", "pol_sequence", "orig_sequence"]:
        if col in good.columns:
            idx_map = {s: i for i, s in enumerate(X_all)}
            mapped = good[col].map(idx_map)
            if mapped.notna().all():
                return groups_all[mapped.astype(int).values], f"matched on '{col}' full-sequence text"
    if len(coords_df) == n_full and len(good) == n_full:
        if "subtype" in good.columns and "subtype" in df.columns:
            match_rate = (good["subtype"].values == df["subtype"].values).mean()
            if match_rate > 0.99:
                return groups_all[good.index.values], f"positional alignment, verified via subtype match ({match_rate:.1%})"
            else:
                print(f"Positional alignment REJECTED: subtype columns only agree {match_rate:.1%} row-by-row.")
        else:
            print("Same row count as the full dataset, but no 'subtype' column in one or both files to "
                  "verify row order -- not risking an unverified positional mapping.")
    print(f"Could not verify a mapping back to CD-HIT-97% clusters. Columns available in "
          f"t11_hxb2_coords.csv: {list(coords_df.columns)}. Send these column names back and the "
          "mapping can be fixed precisely instead of guessed at.")
    return None, "no verified mapping"

amplicon_groups, mapping_method = build_amplicon_groups(coords_df, good, df, X_all, groups_all)
assert amplicon_groups is not None, "Stopping here -- see the message above."
print(f"Mapping method used: {mapping_method}")


### 4. Split helper (with validation carve-out for the CNNs) and shared TF-IDF encoder

In [ ]:

MAX_TFIDF_FEATS = 512
SEEDS = [0, 1, 2]
EPOCHS = 120     # matches the original per-architecture notebooks (was 60 -- bug, now fixed)
BATCH  = 48

def split_grouped_with_val(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte


### 5. SVM + RF, PR alone and RT alone, 3 seeds -- CPU only

In [ ]:

rows = []
pred_rows = []
splits_cache = {}

for col in INPUTS:
    for seed in SEEDS:
        tr, va, te = split_grouped_with_val(good.index.values, y_good, amplicon_groups, seed)
        splits_cache[(col, seed)] = (tr, va, te)
        Xtr, Xva, Xte = fit_tfidf(good.iloc[tr][col].values, good.iloc[va][col].values, good.iloc[te][col].values)

        Xtrva = np.vstack([Xtr.toarray(), Xva.toarray()])
        ytrva = np.concatenate([y_good[tr], y_good[va]])
        yte = y_good[te]
        for model_name, clf in [
            ("svm", LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)),
            ("rf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)),
        ]:
            clf.fit(Xtrva, ytrva)
            pred = clf.predict(Xte)
            rows.append(dict(input=col, seed=seed, model=model_name,
                              f1=f1_score(yte, pred, average="macro"),
                              bal_acc=balanced_accuracy_score(yte, pred)))
            pred_rows.extend([dict(input=col, seed=seed, model=model_name,
                                    test_idx=int(good.index.values[i]), y_true=int(t), y_pred=int(p))
                               for i, t, p in zip(te, yte, pred)])
    pd.DataFrame(rows).to_csv(f"{OUT}/t22_classical_raw.csv", index=False)
    pd.DataFrame(pred_rows).to_csv(f"{OUT}/t22_classical_predictions.csv", index=False)
    print(f"[classical] input={col} done")


### 6. CNNs, PR alone and RT alone, 3 seeds -- needs GPU. Same architectures/training loop as T17/T18/T19/T21.

In [ ]:


class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(input_len, n_classes):
    inp = Input((input_len, 1)); x = GaussianNoise(0.05)(inp)
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for f, sp in [(32, .95), (32, .92)]:  x = res_block(x, f, 1, sp)
    for f, sp in [(64, .88), (64, .85)]:  x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(128, .80), (128, .78)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(256, .75), (256, .72)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def build_vgg16(input_len, n_classes):
    def conv_block(x, f, n_convs):
        for _ in range(n_convs):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((input_len, 1))
    x = conv_block(inp, 32, 2)
    x = conv_block(x,   64, 2)
    x = conv_block(x,  128, 3)
    x = conv_block(x,  256, 3)
    x = conv_block(x,  256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

BUILDERS = {"resnet18": build_resnet18, "vgg16": build_vgg16, "inception": build_inception}
DEEP_MODELS = ["resnet18", "vgg16", "inception"]

def run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_classes):
    """Exact training recipe from the original per-architecture notebooks (verified byte-identical
    in Phase A, same code already used in T18/T24/T25): per-feature StandardScaler (fit on train
    only), one-hot labels, CategoricalCrossentropy(label_smoothing=0.1), Adam(3e-4), up to 120
    epochs, EarlyStopping(patience=15) + ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-6)."""
    tf.random.set_seed(seed)
    sc = StandardScaler()
    Xtr_d = sc.fit_transform(np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr))[..., None]
    Xva_d = sc.transform(np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva))[..., None]
    Xte_d = sc.transform(np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte))[..., None]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model = BUILDERS[model_name](Xtr_d.shape[1], n_classes)
    model.fit(Xtr_d, to_categorical(ytr, n_classes),
              validation_data=(Xva_d, to_categorical(yva, n_classes)),
              epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
              callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                         ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = model.predict(Xte_d, verbose=0)
    pred = prob.argmax(1)
    f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
    tf.keras.backend.clear_session()
    return f1, bal, pred

DEEP_RAW_PATH = f"{OUT}/t22_deep_raw.csv"
if os.path.exists(DEEP_RAW_PATH):
    _existing_deep = pd.read_csv(DEEP_RAW_PATH)
    deep_done_keys = set(zip(_existing_deep["input"], _existing_deep["seed"], _existing_deep["model"]))
    deep_rows = _existing_deep.to_dict("records")
    print(f"Resuming deep models: {len(deep_rows)} rows already done.")
else:
    deep_rows = []
    deep_done_keys = set()

deep_pred_rows = []
for (col, seed), (tr, va, te) in splits_cache.items():
    remaining = [m for m in DEEP_MODELS if (col, seed, m) not in deep_done_keys]
    if not remaining:
        continue
    Xtr, Xva, Xte = fit_tfidf(good.iloc[tr][col].values, good.iloc[va][col].values, good.iloc[te][col].values)
    ytr, yva, yte = y_good[tr], y_good[va], y_good[te]
    for m in remaining:
        f1, bal, pred = run_deep_original(m, Xtr, ytr, Xva, yva, Xte, yte, seed, n_cls)
        deep_rows.append(dict(input=col, seed=seed, model=m, f1=f1, bal_acc=bal))
        deep_pred_rows.extend([dict(input=col, seed=seed, model=m,
                                     test_idx=int(good.index.values[i]), y_true=int(t), y_pred=int(p))
                                for i, t, p in zip(te, yte, pred)])
        pd.DataFrame(deep_rows).to_csv(DEEP_RAW_PATH, index=False)   # incremental save
        print(f"[deep] input={col} seed={seed} model={m} f1={f1:.4f} done")
    pd.DataFrame(deep_pred_rows).to_csv(f"{OUT}/t22_deep_predictions.csv", index=False)

t22_all = pd.concat([pd.DataFrame(rows), pd.DataFrame(deep_rows)], ignore_index=True)
t22_all_predictions = pd.concat([pd.DataFrame(pred_rows), pd.DataFrame(deep_pred_rows)], ignore_index=True)
t22_all.to_csv(f"{OUT}/t22_all_raw.csv", index=False)
t22_all_predictions.to_csv(f"{OUT}/t22_all_predictions.csv", index=False)
print(t22_all.groupby(["input","model"])[["f1","bal_acc"]].agg(["mean","std"]).round(4))


### 7. Zip outputs

In [ ]:
!cd /workspace/outputs/t22 && zip -r t22_pr_rt_alone_outputs.zip t22_all_raw.csv t22_all_predictions.csv t22_classical_raw.csv t22_classical_predictions.csv t22_deep_raw.csv t22_deep_predictions.csv 2>/dev/null; ls -la /workspace/outputs/t22/*.zip